# 选修E10 · Day 2 上机：Agent商业模式设计--从AaaS到outcome-based pricing

**版本**：v5.0 学习材料包
**配套**：notes.md（讲义）｜ data/README.md（真实库/数据）｜ solution.ipynb（参考答案，做完再看）

## 学习目标
学完你能：
1. 用 **pydantic** 定义四种Agent商业模式定价契约（AaaS订阅/按调用计费/outcome-based/分润），实现结构化输出
2. 用 **numpy-financial** 计算三种定价模式12月现金流NPV/IRR，量化推理成本对利润率的影响
3. 用 **statsmodels** 拟合定价弹性回归（log-log OLS），找最优定价点
4. 解释Agent商业模式五阶段演进（按席位→按用量→按任务→按结果→按价值分成），能设计营销Agent混合定价
5. 建立天道推演×商业模式沙盘同构认知--用三时间线推演不同定价模式在推理成本下降/MCP协议标准化下的演化走向

## 真实库与真实数据
- **pydantic**（schema验证）：https://github.com/pydantic/pydantic
- **numpy-financial**（NPV/IRR）：https://github.com/numpy/numpy-financial
- **statsmodels**（弹性回归）：https://github.com/statsmodels/statsmodels
- **真实Agent定价案例**：Cursor/Devin/Intercom Fin/Sierra/11x.ai/DevRev/GitHub Copilot/ChatGPT Plus
- **真实推理成本**：GPT-4o $5/1M / Claude Sonnet $3/1M / DeepSeek V3 $0.27/1M

> 所有库与数据均来自官方公开源，不需要API Key。

## 0. 环境准备

首次运行需安装依赖（取消注释执行一次）：

> 所有库（pydantic/numpy-financial/statsmodels/pandas/matplotlib/numpy）均为本地可用库，不需要API Key。

In [ ]:
# !pip install pydantic numpy-financial statsmodels pandas matplotlib numpy -q

import warnings
warnings.filterwarnings('ignore')

from pydantic import BaseModel, Field, model_validator
import numpy_financial as npf
import statsmodels.api as sm
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from typing import Literal, Optional

print("✓ pydantic, numpy-financial, statsmodels, pandas, matplotlib, numpy 已就绪")
print("  pydantic: Agent商业模式schema契约")
print("  numpy-financial: 三种定价模式NPV/IRR对比")
print("  statsmodels: 定价弹性回归+最优定价点")

## 1. 真实Agent商业模式参数

本Day的参数基于真实Agent定价案例和推理成本基准：

| 参数 | 值 | 真实来源 |
|------|-----|---------|
| AaaS月费 | $200/月 | 营销Agent中等定价（高于Cursor $20，低于Devin $500） |
| 按调用计费 | $0.05/调用 × 4000次/月 | OpenAI API模式扩展 |
| outcome-based | $10/转化 × 150次/月 | Intercom Fin $0.99/解决的高价值版本 |
| GPT-4o推理成本 | $5/1M input tokens | OpenAI 2024-2025定价页 |
| DeepSeek V3推理成本 | $0.27/1M input tokens | DeepSeek官方定价页 |
| 推理token/调用 | 1000 tokens | Agent推理合理消耗 |
| 月增长率 | 8% | Agent产品早期增长 |
| 月贴现率 | 0.10/12 | 年化10%贴现率 |

In [ ]:
# 真实Agent商业模式参数（可追溯来源）
# 来源1: Cursor定价 https://cursor.com/pricing ($20-40/月)
# 来源2: Devin定价 https://devin.ai ($500/月)
# 来源3: Intercom Fin https://www.intercom.com/pricing ($0.99/解决)
# 来源4: OpenAI定价 https://openai.com/api/pricing/ (GPT-4o $5/1M)
# 来源5: DeepSeek定价 https://api-docs.deepseek.com/quick_start/pricing ($0.27/1M)

# === 三种定价模式参数 ===
AAAS_MONTHLY_FEE = 200.0          # AaaS订阅月费 $200
PERCALL_PRICE = 0.05              # 按调用计费 $0.05/调用
PERCALL_MONTHLY_VOLUME = 4000     # 月调用量 4000次
OUTCOME_PRICE = 10.0              # outcome-based $10/转化
OUTCOME_MONTHLY_COUNT = 150       # 月转化数 150次

# === 推理成本基准 ===
TOKENS_PER_CALL = 1000            # 每次Agent调用推理token消耗
INFERENCE_COSTS = {
    "GPT-4o":          5.00 / 1_000_000 * TOKENS_PER_CALL,   # $0.005/调用
    "Claude Sonnet 4": 3.00 / 1_000_000 * TOKENS_PER_CALL,   # $0.003/调用
    "DeepSeek V3":     0.27 / 1_000_000 * TOKENS_PER_CALL,   # $0.00027/调用
}

# === 财务参数 ===
MONTHLY_GROWTH_RATE = 0.08        # 月增长率 8%
MONTHLY_DISCOUNT_RATE = 0.10 / 12 # 月贴现率（年化10%）
FORECAST_MONTHS = 12              # 预测12个月

print("=== 真实Agent商业模式参数 ===")
print(f"AaaS订阅: ${AAAS_MONTHLY_FEE}/月")
print(f"按调用计费: ${PERCALL_PRICE}/调用 × {PERCALL_MONTHLY_VOLUME}次/月 = ${PERCALL_PRICE*PERCALL_MONTHLY_VOLUME}/月")
print(f"outcome-based: ${OUTCOME_PRICE}/转化 × {OUTCOME_MONTHLY_COUNT}次/月 = ${OUTCOME_PRICE*OUTCOME_MONTHLY_COUNT}/月")
print()
print("=== 推理成本基准（每次Agent调用） ===")
for model, cost in INFERENCE_COSTS.items():
    print(f"  {model}: ${cost:.5f}/调用")

## 2. TODO 1：pydantic四种定价模式schema定义

**四种Agent商业模式定价契约**：

| 模式 | 字段 | 计费逻辑 |
|------|------|---------|
| AaaS订阅 | price_per_month | 固定月费 |
| 按调用计费 | price_per_call, monthly_calls | 单价 × 调用量 |
| outcome-based | price_per_outcome, monthly_outcomes | 单价 × 结果数 |
| 分润 | share_pct, baseline_revenue | 增量收入 × 分润比例 |

**要求**：
- 用pydantic BaseModel定义四种定价模式
- 每种模式实现 `monthly_revenue()` 方法计算月收入
- 每种模式实现 `to_contract()` 方法导出结构化输出（Agent可发现的能力声明）
- 用 `@model_validator` 验证字段约束（价格非负、分润比例0-1）

**理论连接**：pydantic schema不仅是数据验证，更是API Economy 2.0的"Agent可发现能力声明"--Agent通过读取其他Agent的schema自动判断能否调用。

In [ ]:
# TODO 1：pydantic四种定价模式schema定义
# 提示：继承BaseModel
#   AaaSSubscription: price_per_month > 0
#   PerCallPricing: price_per_call > 0, monthly_calls >= 0
#   OutcomeBasedPricing: price_per_outcome > 0, monthly_outcomes >= 0
#   RevenueShare: share_pct (0,1), baseline_revenue >= 0
#   每种实现 monthly_revenue() 和 to_contract() 方法
#   用 @model_validator 验证约束

# ===== 你的代码 =====

# raise NotImplementedError

## 3. TODO 2：真实Agent定价案例数据加载与探索

**真实Agent定价案例数据**（来自各产品官方定价页，2025-2026）：

| Agent产品 | 定价模式 | 价格 | 目标市场 |
|----------|---------|------|---------|
| Cursor Pro | AaaS订阅 | $20/月 | 开发者 |
| Cursor Business | AaaS订阅 | $40/月/用户 | 企业开发 |
| Devin | AaaS订阅+任务 | $500/月 | 企业工程 |
| GitHub Copilot | AaaS订阅 | $10-39/月 | 开发者 |
| ChatGPT Plus | AaaS订阅 | $20/月 | 通用 |
| Intercom Fin | outcome-based | $0.99/解决 | 企业客服 |
| Sierra | outcome-based | 按解决率 | 企业客服 |
| 11x.ai | outcome-based | 按预约会议 | 企业销售 |
| DevRev | outcome-based | 按工单解决 | 企业客服 |

**要求**：
- 加载真实Agent定价案例数据到pandas DataFrame
- `df.describe()` 查看数值列统计
- `df.groupby('pricing_model')['price'].agg(['mean','min','max'])` 对比定价模式
- `df.groupby('target_market')['price'].mean()` 对比目标市场

**理论连接**：真实定价案例跨数量级（$0.99 ~ $500），反映Agent商业模式多样性。

In [ ]:
# TODO 2：真实Agent定价案例数据加载与探索
# 提示：将真实案例数据加载到DataFrame
#   列: product, pricing_model, price, unit, target_market
#   df.describe() 查看统计
#   df.groupby('pricing_model')['price'].agg(['mean','min','max']) 对比定价模式
#   df.groupby('target_market')['price'].mean() 对比目标市场
# 要求：打印数据表、描述统计、按定价模式分组、按目标市场分组

# ===== 你的代码 =====

# raise NotImplementedError

## 4. TODO 3：三种定价模式12月现金流NPV/IRR对比

用 **numpy-financial** 计算三种定价模式（AaaS订阅/按调用计费/outcome-based）的12月现金流NPV/IRR。

**建模假设**：
- 月增长率 8%（Agent产品早期增长）
- 月贴现率 0.10/12（年化10%）
- 推理成本：每次Agent调用消耗1000 tokens，按GPT-4o $5/1M计
- 三种模式有不同的月收入和月调用量

| 模式 | 月收入 | 月调用量 | 推理成本/月 |
|------|--------|---------|------------|
| AaaS订阅 | $200 | 1000次（客户用法不限量，但实际用量） | $5 |
| 按调用计费 | $0.05 × 4000 = $200 | 4000次 | $20 |
| outcome-based | $10 × 150 = $1500 | 3000次（多轮交互达成转化） | $15 |

**要求**：
- 建模12月现金流（收入 - 推理成本）
- 用 `npf.npv(rate, cashflows)` 计算NPV
- 用 `npf.irr(cashflows)` 计算IRR
- 对比三种模式的财务表现

**理论连接**：推理成本是Agent商业模式与传统SaaS的本质区别--传统SaaS边际成本接近零，Agent每次调用都消耗token。

In [ ]:
# TODO 3：三种定价模式12月现金流NPV/IRR对比
# 提示：
#   1. 三种模式的monthly_revenue和monthly_calls不同
#   2. inference_cost = monthly_calls * INFERENCE_COSTS["GPT-4o"]
#   3. 12月现金流，每月增长8%（收入和调用量同步增长）
#   4. npf.npv(MONTHLY_DISCOUNT_RATE, cashflows) 计算NPV
#   5. npf.irr(cashflows) 计算IRR
# 要求：计算三种模式的12月NPV/IRR，打印对比表

# ===== 你的代码 =====

# raise NotImplementedError

## 5. TODO 4：statsmodels定价弹性回归（log-log OLS）

用 **statsmodels** 拟合定价弹性回归，找最优定价点。

**价格弹性**（Price Elasticity）：价格变化1%时需求变化百分之几
- 弹性 < -1：弹性需求，降价增收
- 弹性 > -1：非弹性需求，涨价增收
- 弹性 = -1：单位弹性，最优定价点附近

**方法**：log-log OLS回归 `log(adopt_rate) ~ log(price)`，斜率即弹性

**数据**：基于真实Agent定价案例的价格点，建模对应的市场采纳率（基于市场定位和竞争强度）

**要求**：
- 准备价格点数组（含真实案例价格）和对应的采纳率
- 取log后用 `sm.OLS(log_q, sm.add_constant(log_p)).fit()` 拟合
- 解读弹性（斜率）、R²、95% CI
- 找最优定价点（利润最大化）

**理论连接**：定价弹性是经济学和营销学的标准方法，statsmodels OLS是计量经济学的标准工具。

In [ ]:
# TODO 4：statsmodels定价弹性回归（log-log OLS）
# 提示：
#   1. 准备价格点和采纳率数据（基于真实案例价格建模）
#   2. log_p = np.log(prices), log_q = np.log(adopt_rates)
#   3. X = sm.add_constant(log_p), model = sm.OLS(log_q, X).fit()
#   4. elasticity = model.params[1] (斜率即弹性)
#   5. 95% CI: model.conf_int().iloc[1]
#   6. R²: model.rsquared
# 要求：拟合弹性回归，打印弹性/R²/p值/95% CI，找最优定价点

# ===== 你的代码 =====

# raise NotImplementedError

## 6. TODO 5：推理成本敏感度分析

分析推理成本下降（GPT-4o -> Claude Sonnet -> DeepSeek V3）对三种定价模式利润率的影响。

**推理成本基准**：
- GPT-4o: $5/1M input tokens -> $0.005/调用
- Claude Sonnet 4: $3/1M -> $0.003/调用
- DeepSeek V3: $0.27/1M -> $0.00027/调用（降低95%）

**要求**：
- 对三种定价模式 × 三种推理成本，计算12月总利润
- 计算利润率（利润/收入）
- 找出"推理成本下降使outcome-based从亏到盈"的阈值

**理论连接**：推理成本下降是outcome-based pricing可行的关键条件。DeepSeek V3比GPT-4o低95%，使高调用量的outcome-based模式利润率大幅提升。

In [ ]:
# TODO 5：推理成本敏感度分析
# 提示：
#   1. 对三种定价模式 × 三种推理成本（GPT-4o/Claude/DeepSeek）
#   2. 用build_cashflows计算12月总利润和利润率
#   3. 利润率 = 总利润 / 总收入
#   4. 打印3x3矩阵，找盈亏平衡阈值
# 要求：计算9种组合的利润率，打印矩阵，分析推理成本影响

# ===== 你的代码 =====

# raise NotImplementedError

## 7. TODO 6：matplotlib可视化（4个子图）

用matplotlib绘制4个子图：

1. **三模式NPV对比**（柱状图）：三种定价模式在GPT-4o推理成本下的12月NPV
2. **推理成本对利润率影响**（折线图）：三种定价模式在三种推理成本下的利润率
3. **定价弹性曲线**（散点+回归线）：log-log弹性回归的可视化
4. **最优定价利润曲线**（曲线图）：不同价格下的利润，标注最优点

**理论连接**：可视化让Agent商业模式的财务对比直观可见，是天道推演沙盘的"局势可视化"。

In [ ]:
# TODO 6：matplotlib可视化（4个子图）
# 提示：
#   1. plt.subplots(2,2,figsize=(14,10))
#   2. 子图1: ax.bar(三模式NPV)
#   3. 子图2: ax.plot(推理成本 vs 利润率, 三条线)
#   4. 子图3: ax.scatter(log_p, log_q) + ax.plot(回归线)
#   5. 子图4: ax.plot(价格 vs 利润) + ax.axvline(最优价格)
# 要求：4个子图都有标题、标签、数据

# ===== 你的代码 =====

# raise NotImplementedError

## 8. 天道推演 × 商业模式沙盘

本Day的商业模式设计本质是**商业版的天道推演沙盘**：

| 天道推演能力 | 商业模式设计对应 | 产出 |
|-------------|----------------|------|
| 局势感知 | 真实Agent定价案例 + 推理成本基准 | 市场基线 |
| 因果链追踪 | 定价 → 采纳率 → 收入 → 利润 | 财务模型 |
| 沙盘模拟（3层推演） | 12月NPV/IRR + 弹性 + 推理成本敏感度 | 三时间线推演 |
| 概率评估 | 弹性回归95% CI + 推理成本矩阵 | 风险量化 |
| 最优路径推荐 | 三种定价模式对比 + 最优定价点 | 策略选择 |

### 三时间线推演

- **immediate（月）**：单月现金流，推理成本对当月利润的硬约束
- **near（年，12月）**：NPV/IRR，考虑增长率和贴现率的财务可行性
- **far（3年+）**：推理成本下降趋势 + MCP协议标准化 + A2A经济兴起

### 2026-2028范式转移预判

| 时间 | 推理成本 | 主流定价 | 触发条件 |
|------|---------|---------|---------|
| 2026 | $5/1M (GPT-4o) | AaaS订阅为主 | 推理成本高，outcome-based难盈利 |
| 2027 | $0.5/1M (DeepSeek级) | outcome-based兴起 | 推理成本下降90%，按结果计费可行 |
| 2028 | $0.05/1M | 分润模式普及 | A2A经济成熟，MCP协议标准化 |

---

## 9. 作业与评估

- [ ] 完成 `starter.ipynb`（6个TODO全部填好）
- [ ] 三种定价模式NPV/IRR对比有数据
- [ ] 弹性回归有显著结果（p<0.05）
- [ ] 4个子图有数据
- [ ] 一段300字分析：三种定价模式在你的营销场景下，哪种最优？推理成本下降如何改变选择？

---

*本笔记本由v5.0学习材料包升级生成。理论部分引用独立教材，上机部分用真实库（pydantic+numpy-financial+statsmodels+pandas+matplotlib+numpy）+ TODO脚手架，定价案例和推理成本基于真实公开数据。*